8. Simulate the BB84 Quantum Key Distribution protocol using Qiskit circuits in Colab.

In [1]:
!pip install qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.3 MB/s eta 0:00:00


In [8]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import Aer

num_bits = 16

alice_bits = np.random.randint(2, size=num_bits)
alice_bases = np.random.randint(2, size=num_bits)
bob_bases = np.random.randint(2, size=num_bits)

def encode_message(bits, bases):
    message = []
    for i in range(len(bits)):
        qc = QuantumCircuit(1, 1)
        if bits[i] == 1:
            qc.x(0)
        if bases[i] == 1:
            qc.h(0)
        message.append(qc)
    return message

def measure_message(message, bases):
    backend = Aer.get_backend('qasm_simulator')
    measurements = []
    for i in range(len(bases)):
        qc = message[i]
        if bases[i] == 1:
            qc.h(0)
        qc.measure(0, 0)
        result = backend.run(qc, shots=1, memory=True).result()
        measured_bit = int(result.get_memory()[0])
        measurements.append(measured_bit)
    return measurements

def remove_garbage(a_bases, b_bases, bits):
    good_bits = []
    for i in range(len(a_bases)):
        if a_bases[i] == b_bases[i]:
            good_bits.append(bits[i])
    return good_bits

message = encode_message(alice_bits, alice_bases)

bob_results = measure_message(message, bob_bases)

alice_key = remove_garbage(alice_bases, bob_bases, alice_bits)
bob_key = remove_garbage(alice_bases, bob_bases, bob_results)

print(f"Alice's initial bits: {alice_bits}")
print(f"Alice's bases:        {alice_bases}")
print(f"Bob's bases:          {bob_bases}")
print(f"Bob's results:        {bob_results}")
print("-" * 50)
print(f"Alice's sifted key:   {alice_key}")
print(f"Bob's sifted key:     {bob_key}")
print(f"Keys match:           {alice_key == bob_key}")

Alice's initial bits: [0 0 0 0 0 1 0 1 1 1 0 0 1 1 1 0]
Alice's bases:        [1 0 1 0 1 1 0 1 1 0 0 1 1 0 0 1]
Bob's bases:          [0 0 1 0 1 0 0 1 0 0 0 1 1 0 0 1]
Bob's results:        [1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0]
--------------------------------------------------
Alice's sifted key:   [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0)]
Bob's sifted key:     [0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0]
Keys match:           True
